# Global model — train + compare + pick winner

Trains **one** model across all (student, course) pairs instead of 62
per-course classifiers. Compares Logistic Regression, Random Forest, and
HistGradientBoosting; picks the winner by per-course count MAPE (the
metric the dashboard cares about), not just AUC.

**Outputs** under `ml/artifacts/`:
- `global_model.pkl` — winning model + scaler + ohe + meta in one bundle
- `global_model_meta.json` — comparison numbers, threshold, feature schema

**Inputs**: `htu_enrollments.csv` and `htu_students.csv` (the same files the
existing per-course pipeline uses; the notebook auto-locates them across
the usual paths).

## Cell 1: Imports & config

In [14]:
import json
import os
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import (
    HistGradientBoostingClassifier, RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ── Paths ───────────────────────────────────────────────────────────────
# Notebook is expected to live in ml/ alongside inference.py and train_pipeline.py.
# Data + artifacts live where the existing training pipeline already puts them.
ROOT = Path.cwd()                   # notebook's working dir = ml/
DATA_DIR = ROOT / "Data" / "output" # ml/Data/output/htu_*.csv
ART_DIR  = ROOT / "artifacts"       # ml/artifacts/
ART_DIR.mkdir(parents=True, exist_ok=True)

assert (DATA_DIR / "htu_enrollments.csv").exists(), (
    f"Expected htu_enrollments.csv in {DATA_DIR}. "
    "If your notebook is somewhere other than ml/, set ROOT manually."
)
print(f"Reading data from: {DATA_DIR}")
print(f"Writing artifacts to: {ART_DIR}")

# Reproducibility
RNG = np.random.RandomState(42)
TEST_SEMESTER_RANK = 1   # use last 1 semester as test (next-most-recent)

Reading data from: c:\Users\saeed\Desktop\capstone\Backend\Gear\ml\Data\output
Writing artifacts to: c:\Users\saeed\Desktop\capstone\Backend\Gear\ml\artifacts


## Cell 2: Load CSVs

In [15]:
students = pd.read_csv(DATA_DIR / "htu_students.csv",
                       dtype={"student_id": str})
enrolls  = pd.read_csv(DATA_DIR / "htu_enrollments.csv",
                       dtype={"student_id": str, "course_code": str})

# Standardize column names that may vary
if "plan_key" not in students.columns:
    students["plan_key"] = students["major"] + "_" + students["degree_type"]
enrolls["course_code"] = enrolls["course_code"].str.zfill(10)

# Chronological key: Fall=1, Spring=2, Summer=3 within an academic year
TERM_RANK = {"Fall": 1, "Spring": 2, "Summer": 3}
enrolls["sem_key"] = (
    enrolls["year"].astype(int) * 10
    + enrolls["term"].map(TERM_RANK).fillna(1).astype(int)
)
enrolls["passed"] = enrolls["grade"].isin(
    ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D+", "D", "P", "M"]
)

# Sort once — every other step assumes chronological order per student
enrolls = enrolls.sort_values(["student_id", "sem_key"]).reset_index(drop=True)

print(f"Loaded {len(students):,} students, {len(enrolls):,} enrollments")
print(f"Semester range: {enrolls['semester_label'].min()} → "
      f"{enrolls['semester_label'].max()}")

Loaded 3,000 students, 91,165 enrollments
Semester range: 2019_Fall → 2025_Fall


## Cell 3: Build per-student snapshots

In [16]:
# A "snapshot" is the student's state at the end of some semester S, plus
# the set of courses they took in S+1 (the prediction target).
#
# We iterate semester-by-semester. After each semester S finishes, we
# record one snapshot per active student. Their next-semester course set
# is harvested from S+1's enrollments.

def build_snapshots(enrolls_df, students_df):
    snapshots = []   # list of dicts
    next_courses = []  # list of sets

    # Index enrollments by student for fast lookup
    by_student = {sid: g for sid, g in enrolls_df.groupby("student_id",
                                                          sort=False)}

    student_meta = students_df.set_index("student_id").to_dict("index")

    all_sem_keys = sorted(enrolls_df["sem_key"].unique())

    for sid, g in by_student.items():
        if sid not in student_meta:
            continue
        meta_row = student_meta[sid]
        unique_sems = g["sem_key"].drop_duplicates().tolist()
        if len(unique_sems) < 2:
            continue   # need at least one semester of history + one future

        for i, sk in enumerate(unique_sems[:-1]):
            past = g[g["sem_key"] <= sk]
            future_sk = unique_sems[i + 1]
            future = g[g["sem_key"] == future_sk]

            passed_set = set(past[past["passed"]]["course_code"])
            failed_set = set(past[~past["passed"]]["course_code"])

            cum_hrs = int(past[past["passed"]]["credit_hours"].sum())
            cum_gpa = float(past["cum_gpa"].iloc[-1]) if len(past) else 0.0

            snapshots.append({
                "student_id":     sid,
                "snapshot_key":   sk,
                "next_sem_key":   future_sk,
                "next_sem_term":  future["term"].iloc[0],
                "plan_key":       meta_row["plan_key"],
                "major":          meta_row.get("major", ""),
                "degree_type":    meta_row.get("degree_type", "Bachelor"),
                "admission_year": int(meta_row.get("admission_year", 2020)),
                "rem_arabic":     int(bool(meta_row.get("needs_rem_arabic", 0))),
                "rem_english":    int(bool(meta_row.get("needs_rem_english", 0))),
                "rem_math":       int(bool(meta_row.get("needs_rem_math", 0))),
                "cum_hrs":        cum_hrs,
                "cum_gpa":        cum_gpa,
                "n_passed":       len(passed_set),
                "n_failed":       len(failed_set),
                "current_year":   max(1, min(4, cum_hrs // 33 + 1)),
                "passed_set":     passed_set,
                "failed_set":     failed_set,
            })
            next_courses.append(set(future["course_code"]))

    snap_df = pd.DataFrame(snapshots)
    snap_df["next_courses"] = next_courses
    return snap_df


print("Building snapshots…")
t0 = time.time()
snap_df = build_snapshots(enrolls, students)
print(f"  {len(snap_df):,} snapshots in {time.time()-t0:.1f}s")

Building snapshots…
  19,057 snapshots in 45.6s


## Cell 4: Course-side features

In [17]:
# For each course we summarize: department code (first 3 digits), credit
# hours, the empirical "typical year" students take it (median current_year
# at enrollment time — robust to plan variations), historical enrollment
# trend, and the term it's typically offered in.

print("Building course features…")
course_meta = enrolls.groupby("course_code").agg(
    course_name=("course_name", lambda s: s.mode().iat[0]
                 if not s.mode().empty else ""),
    course_credit_hrs=("credit_hours", "median"),
).reset_index()
course_meta["dept_code"] = course_meta["course_code"].str[:3]

# Estimate "typical year" for each course: at what current_year do students
# usually take it? This replaces the plan-map RECOMMENDED_YEARS dict.
# We use snapshots so the year is taken at the moment of decision.
exp_rows = []
for _, r in snap_df.iterrows():
    for c in r["next_courses"]:
        exp_rows.append({"course_code": c, "current_year": r["current_year"]})
course_year_df = pd.DataFrame(exp_rows)
course_year = (
    course_year_df.groupby("course_code")["current_year"]
    .median().reset_index()
    .rename(columns={"current_year": "course_typical_year"})
)
course_meta = course_meta.merge(course_year, on="course_code", how="left")
course_meta["course_typical_year"] = course_meta["course_typical_year"].fillna(2)

# Historical avg per-semester count + recent trend
sem_counts = (
    enrolls.groupby(["course_code", "sem_key"])["student_id"]
    .nunique().reset_index(name="n_students")
)
hist_avg = (
    sem_counts.groupby("course_code")["n_students"].mean()
    .reset_index().rename(columns={"n_students": "course_hist_avg"})
)
course_meta = course_meta.merge(hist_avg, on="course_code", how="left")
course_meta["course_hist_avg"] = course_meta["course_hist_avg"].fillna(0)

# Term affinity: what fraction of historical offerings landed in each term
term_share = (
    enrolls.groupby(["course_code", "term"])["student_id"].nunique()
    .unstack(fill_value=0)
)
term_share = term_share.div(term_share.sum(axis=1).replace(0, 1), axis=0)
term_share.columns = [f"course_term_share_{c}" for c in term_share.columns]
course_meta = course_meta.merge(term_share.reset_index(), on="course_code",
                                how="left").fillna(0)

print(f"  {len(course_meta):,} courses with features: "
      f"{[c for c in course_meta.columns if c != 'course_code']}")

Building course features…
  97 courses with features: ['course_name', 'course_credit_hrs', 'dept_code', 'course_typical_year', 'course_hist_avg', 'course_term_share_Fall', 'course_term_share_Spring', 'course_term_share_Summer']


## Cell 5: Build (snapshot × course) training pairs

In [18]:
# The "decision space" for each snapshot is courses in the student's plan
# they haven't passed yet. Positives: courses taken in next semester.
# Negatives: courses they could have taken but didn't.
#
# This is the key difference from the per-course approach — we never train
# on (student, course) pairs that don't represent a real decision.

# Plan-membership: courses ever taken by students sharing the same plan_key.
plan_courses = (
    enrolls.merge(students[["student_id", "plan_key"]], on="student_id")
    .groupby("plan_key")["course_code"].apply(set).to_dict()
)

print("Building training pairs…")
t0 = time.time()
pair_rows = []
for _, snap in snap_df.iterrows():
    plan_set = plan_courses.get(snap["plan_key"], set())
    candidates = plan_set - snap["passed_set"]   # not already passed
    pos = snap["next_courses"] & candidates
    neg = candidates - snap["next_courses"]

    # Keep ALL positives AND ALL negatives. Subsampling negatives biases the
    # model toward over-predicting positives at inference time, where every
    # candidate is scored without subsampling.
    for c in pos:
        pair_rows.append((snap.name, c, 1))
    for c in neg:
        pair_rows.append((snap.name, c, 0))

pairs_df = pd.DataFrame(pair_rows, columns=["snap_idx", "course_code", "y"])
print(f"  {len(pairs_df):,} pairs ({pairs_df['y'].mean():.1%} positive) "
      f"in {time.time()-t0:.1f}s")

Building training pairs…
  578,472 pairs (11.9% positive) in 2.0s


## Cell 6: Assemble feature matrix

In [19]:
# Join student-side features (per snapshot) with course-side features (per
# course) and add a few interaction features that should be most useful:
#   - year_distance: |current_year - course_typical_year|
#   - already_failed: did the student fail this course before?

snap_keep = [
    "snapshot_key", "next_sem_key", "next_sem_term",
    "plan_key", "major", "degree_type",
    "admission_year", "rem_arabic", "rem_english", "rem_math",
    "cum_hrs", "cum_gpa", "n_passed", "n_failed", "current_year",
]
snap_features = snap_df[snap_keep].copy()
snap_features.index.name = "snap_idx"

# Per-pair already-failed lookup (vectorized via a set lookup)
failed_lookup = snap_df["failed_set"].to_dict()
pairs_df["already_failed"] = pairs_df.apply(
    lambda r: int(r["course_code"] in failed_lookup[r["snap_idx"]]), axis=1
)

# Merge all sides
df = pairs_df.merge(snap_features, left_on="snap_idx", right_index=True)
df = df.merge(course_meta, on="course_code", how="left")
df["year_distance"] = (df["current_year"] - df["course_typical_year"]).abs()

# Categorical / numeric column groups
CAT_COLS = ["plan_key", "major", "degree_type", "next_sem_term", "dept_code"]
NUM_COLS = [
    "admission_year", "rem_arabic", "rem_english", "rem_math",
    "cum_hrs", "cum_gpa", "n_passed", "n_failed", "current_year",
    "course_credit_hrs", "course_typical_year", "course_hist_avg",
    "year_distance", "already_failed",
] + [c for c in df.columns if c.startswith("course_term_share_")]

print(f"Final dataset: {len(df):,} rows, "
      f"{len(CAT_COLS) + len(NUM_COLS)} features "
      f"({len(CAT_COLS)} categorical, {len(NUM_COLS)} numeric)")

Final dataset: 578,472 rows, 22 features (5 categorical, 17 numeric)


## Cell 7: Temporal train/test split

In [20]:
# Hold out the most recent prediction window so we can evaluate on data
# the model has never seen. "snapshot_key" is the semester the snapshot
# was taken at — sort, take last K, those rows form the test set.

unique_sk = sorted(df["snapshot_key"].unique())
test_keys = set(unique_sk[-TEST_SEMESTER_RANK:])
train_mask = ~df["snapshot_key"].isin(test_keys)

df_train = df[train_mask].reset_index(drop=True)
df_test  = df[~train_mask].reset_index(drop=True)

print(f"Train: {len(df_train):,} rows ({df_train['y'].mean():.1%} positive) — "
      f"snapshots through sem_key {max(s for s in unique_sk if s not in test_keys)}")
print(f"Test:  {len(df_test):,} rows ({df_test['y'].mean():.1%} positive) — "
      f"sem_key in {sorted(test_keys)}")

Train: 553,220 rows (11.7% positive) — snapshots through sem_key 20242
Test:  25,252 rows (15.0% positive) — sem_key in [np.int64(20243)]


## Cell 8: Encode + scale

In [21]:
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ohe.fit(df_train[CAT_COLS].astype(str))

scaler = StandardScaler()
scaler.fit(df_train[NUM_COLS])

def transform(frame):
    cat = ohe.transform(frame[CAT_COLS].astype(str))
    num = scaler.transform(frame[NUM_COLS])
    return np.hstack([cat, num])

X_train = transform(df_train)
y_train = df_train["y"].values
X_test  = transform(df_test)
y_test  = df_test["y"].values

print(f"Feature matrix: {X_train.shape}")

Feature matrix: (553220, 35)


## Cell 9: Train candidate models

In [22]:
# Train HistGradientBoostingClassifier — handles class imbalance natively,
# produces calibrated probabilities, and is fast on the ~600k-pair scale
# we get when we don't subsample negatives. Earlier diagnostic showed LR
# (badly miscalibrated) and RF (similar perf but 4x slower) didn't help.

trained = {}
print("\nTraining HGBT…")
t0 = time.time()
model = HistGradientBoostingClassifier(
    max_iter=400, learning_rate=0.05, max_depth=8, random_state=42,
).fit(X_train, y_train)
trained["hgbt"] = (model, time.time() - t0)
print(f"  fit in {trained['hgbt'][1]:.1f}s")



Training HGBT…
  fit in 31.2s


## Cell 10: Evaluate

In [23]:
# Two-tier evaluation:
#   1. Standard binary metrics (AUC, F1) — sanity check
#   2. Per-course count MAPE on test set — what the dashboard actually cares
#      about. For each course, sum predicted probabilities across test rows
#      and compare to actual count of positives.

def per_course_count_error(df_test_local, y_true, y_score):
    """Returns DataFrame with predicted_count / actual_count per course
    plus aggregate MAE and MAPE."""
    out = df_test_local.assign(score=y_score, y_true_local=y_true).groupby(
        "course_code"
    ).agg(predicted=("score", "sum"),
          actual=("y_true_local", "sum")).reset_index()
    out["abs_err"] = (out["predicted"] - out["actual"]).abs()
    # MAPE only over courses that had ≥1 actual enrolment to avoid div0
    nz = out[out["actual"] > 0]
    mape = (nz["abs_err"] / nz["actual"]).mean() * 100
    mae = out["abs_err"].mean()
    return out, mae, mape


print("\n" + "=" * 72)
print(f"{'Model':<10} {'AUC':>6} {'F1@0.5':>7} {'Course-MAE':>11} "
      f"{'Course-MAPE':>12} {'Fit time':>10}")
print("-" * 72)

results = {}
for name, (model, fit_time) in trained.items():
    proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, proba)
    f1  = f1_score(y_test, (proba >= 0.5).astype(int))
    per_course, mae, mape = per_course_count_error(df_test, y_test, proba)
    results[name] = {
        "auc": auc, "f1": f1, "course_mae": mae, "course_mape": mape,
        "fit_time": fit_time, "per_course": per_course, "proba": proba,
    }
    print(f"{name:<10} {auc:>6.3f} {f1:>7.3f} {mae:>11.2f} "
          f"{mape:>11.1f}% {fit_time:>9.1f}s")
print("=" * 72)


Model         AUC  F1@0.5  Course-MAE  Course-MAPE   Fit time
------------------------------------------------------------------------
hgbt        0.940   0.629        4.08        21.0%      31.2s


## Cell 11: Pick winner & save artifacts

In [24]:
winner = "hgbt"
best_model = trained[winner][0]
best_proba = results[winner]["proba"]

# Threshold doesn't affect probability-mode predictions but we keep it
# for backwards compatibility with the bundle schema.
best_thr = 0.5
print(f"Winner: {winner}")
print(f"Threshold (probability mode is default in inference): {best_thr}")


Winner: hgbt
Threshold (probability mode is default in inference): 0.5


## Cell 12: Sanity check — predicted vs actual per course on test set

In [25]:
print("\nTop 20 courses by actual test enrollment "
      "(predicted should track actual):")
print("-" * 72)
print(f"{'Course':<12} {'Actual':>8} {'Predicted':>11} {'Diff':>8} "
      f"{'%Err':>7}")
print("-" * 72)
top = results[winner]["per_course"].sort_values("actual",
                                                ascending=False).head(20)
for _, r in top.iterrows():
    diff = r["predicted"] - r["actual"]
    pct = (abs(diff) / r["actual"] * 100) if r["actual"] > 0 else 0.0
    print(f"{r['course_code']:<12} {r['actual']:>8.0f} "
          f"{r['predicted']:>11.1f} {diff:>+8.1f} {pct:>6.1f}%")
print("-" * 72)


Top 20 courses by actual test enrollment (predicted should track actual):
------------------------------------------------------------------------
Course         Actual   Predicted     Diff    %Err
------------------------------------------------------------------------
0010204282        188       169.3    -18.7    9.9%
0040303221        160       156.8     -3.2    2.0%
0000203280        153       145.7     -7.3    4.8%
0040201201        147       146.1     -0.9    0.6%
0000204313        141       146.4     +5.4    3.8%
0000201391        139       121.2    -17.8   12.8%
0040302211        137       132.9     -4.1    3.0%
0040302231        126       125.8     -0.2    0.1%
0010203180        121       130.0     +9.0    7.4%
0030301123        116       122.3     +6.3    5.4%
0040201260        116       117.6     +1.6    1.4%
0040201220        112       110.2     -1.8    1.6%
0040201290        108       114.2     +6.2    5.8%
0010203380        107       101.0     -6.0    5.6%
0040303130    

## Cell 12.5: Clean up old per-course artifacts (optional)

The per-course `.pkl` files under `ml/artifacts/course_models/` are no
longer used by the global model. Delete them so the artifacts directory
doesn't carry stale junk. Comment out this cell if you want to keep them
around as a fallback.

In [26]:
import shutil

obsolete = [
    ART_DIR / "course_models",
    ART_DIR / "shap_explainers",
    ART_DIR / "model_manifest.csv",
    ART_DIR / "all_model_results_checkpoint.pkl",
    ART_DIR / "model_timing_checkpoint.pkl",
    ART_DIR / "all_course_codes.pkl",
    ART_DIR / "course_meta_lookup.pkl",
    ART_DIR / "course_ts_lookup.pkl",
    ART_DIR / "ohe.pkl",
    ART_DIR / "plan_course_map.pkl",
    ART_DIR / "prereq_map.pkl",
    ART_DIR / "inference_meta.json",
]
for p in obsolete:
    if p.is_dir():
        shutil.rmtree(p)
        print(f"Removed dir  {p.name}/")
    elif p.exists():
        p.unlink()
        print(f"Removed file {p.name}")
